In [38]:
!pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable


In [39]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [40]:
from pathlib import Path

print(Path.cwd())

for root, dirs, files in os.walk("."):
    if "train.csv" in files:
        print("FOUND:", root)

/home/studinstru/dlGenAI/dlgenai-t22026/notebooks


## Step 1: Setup & environment check

In [41]:
import sys, platform
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers

import matplotlib.pyplot as plt

print("Python      :", sys.version.split()[0])
print("Platform    :", platform.system(), platform.release())
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("-" * 45)

# The device we'll use for all tensor computation.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device      :", device)
if device == "cuda":
    print("GPU         :", torch.cuda.get_device_name(0))
    print("VRAM (GB)   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print("-" * 45)

# Confirm the dataset files exist (paths are relative to this notebook's folder).
DATA_DIR = Path("../dataset/smart-mcq-solver-challenge")
for fname in ["train.csv", "test.csv", "sample_submission.csv"]:
    fpath = DATA_DIR / fname
    status = "FOUND   " if fpath.exists() else "MISSING!"
    print(f"{fname:22s} {status} -> {fpath}")

#import os
#from kaggle_secrets import UserSecretsClient
#os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

Python      : 3.10.12
Platform    : Linux 6.6.114.1-microsoft-standard-WSL2
NumPy       : 2.2.6
Pandas      : 2.3.3
PyTorch     : 2.5.1+cu121
Transformers: 5.12.0
---------------------------------------------
Device      : cuda
GPU         : NVIDIA GeForce RTX 3050 Laptop GPU
VRAM (GB)   : 4.29
---------------------------------------------
train.csv              FOUND    -> ../dataset/smart-mcq-solver-challenge/train.csv
test.csv               FOUND    -> ../dataset/smart-mcq-solver-challenge/test.csv
sample_submission.csv  FOUND    -> ../dataset/smart-mcq-solver-challenge/sample_submission.csv


## Step 2: Reading inputs

In [42]:
train = pd.read_csv("../dataset/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("../dataset/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("../dataset/smart-mcq-solver-challenge/sample_submission.csv")

sample_sub.to_csv("submission.csv", index=False)

In [43]:
test.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [44]:
train.describe()

,id
count,2000.000000
mean,1000.500000
std,577.494589
min,1.000000
25%,500.750000
50%,1000.500000
75%,1500.250000
max,2000.000000


In [45]:
# check the shape of train and test
print(train.shape)
print(test.shape)
print(sample_sub.shape)

(2000, 8)
(500, 7)
(500, 2)


In [46]:
print("train columns:", list(train.columns))
print("test columns :", list(test.columns))
print()

print("Missing values per column (train):")
print(train.isna().sum().to_string())
print()

train columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
test columns : ['id', 'prompt', 'A', 'B', 'C', 'D', 'E']

Missing values per column (train):
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0



Q1. Calculate the frequency distribution of the correct answer (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [47]:
freq = train["answer"].value_counts()
max_fre = freq.max()
min_fre = freq.min()
#print(freq, max_fre, min_fre)
print("Sum of occurrences of the most frequent option and the least frequent option:", max_fre + min_fre)

Sum of occurrences of the most frequent option and the least frequent option: 814


Q2: After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [48]:
import string

#help(str.maketrans)
#print("String punctuation characters:", string.punctuation)

# Remove punctuation and convert to lowercase
translator = str.maketrans('', '', string.punctuation)

#table = str.maketrans("abc", "xyz")
#text = "apple banana cat"
#print(text.translate(table))

vocab = set() # for unique characters

for text in train["prompt"].astype(str):
    cleaned_text = text.lower().translate(translator)
    vocab.update(cleaned_text.split())

print("Length of unique characters in the vocabulary:", len(vocab))
#print("Unique characters in the vocabulary:", sorted(vocab))


Length of unique characters in the vocabulary: 859


Q3. Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering? 

In [49]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

prompt = train.loc[train["id"]==1, "prompt"].iloc[0]

translator = str.maketrans("", "", string.punctuation)

cleaned_prompt = prompt.lower().translate(translator)
words = cleaned_prompt.split()
filter_words = [word for word in words if word not in ENGLISH_STOP_WORDS]

print(f"Original prompt:\n{prompt}\n")
print(f"Filtered words:\n{filter_words}\n")

print("Count of filtered words:", len(filter_words))

Original prompt:
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.

Filtered words:
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']

Count of filtered words: 13


Q4. Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer? 

In [50]:
from sklearn.feature_extraction.text import TfidfVectorizer

combined_text = (
    train["prompt"].fillna("")  + " "+
    train["A"].fillna("") + " " + 
    train["B"].fillna("") + " " + 
    train["C"].fillna("") + " " + 
    train["D"].fillna("") + " " + 
    train["E"].fillna("")
    )

vectorizer = TfidfVectorizer(stop_words="english")
x = vectorizer.fit_transform(combined_text)

print("Shape:", x.shape)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))


Shape: (2000, 2762)
Vocabulary size: 2762


Q5. Using the TF-IDF vectorizer fitted in Question 4, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [51]:
from sklearn.metrics.pairwise import cosine_similarity

row = train.loc[train["id"] == 1].iloc[0]

prompt = row["prompt"]
option_a = row["A"]


prompt_vec = vectorizer.transform([prompt])
option_a_vec = vectorizer.transform([option_a])

similarity = cosine_similarity(prompt_vec, option_a_vec)[0][0]

print(similarity)
print(round(similarity, 4))

0.27202429519891635
0.272


Q6. Expand the logic from Question 5: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [53]:
options = ["A", "B", "C", "D", "E"]

correct = 0

for _, row in train.iterrows():
    
    prompt_vec = vectorizer.transform([str(row["prompt"])])

    scores = []

    for opt in options:
        score = cosine_similarity(prompt_vec, vectorizer.transform([str(row[opt])]))[0][0]
        scores.append(score)

    predicted = options[scores.index(max(scores))]

    if predicted == row["answer"]:
        correct += 1

accuracy = correct / len(train)*100
print(f"Accuracy: {accuracy:.2f}%") 

Accuracy: 13.55%


Q7. If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B? (Mean Average Precision at 3.)

In [54]:
def ap_at_3(actual, predicted):
    if actual in predicted[:3]:
        return 1 / (predicted[:3].index(actual) + 1)
    return 0

actual = "C"
predicted = ["C", "A", "B"]

print(f"AP@3: {ap_at_3(actual, predicted):.2f}")

AP@3: 1.00


Q8. If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  

In [55]:
actual = "B"
predicted = ["D", "B", "E"]

print(f"AP@3: {ap_at_3(actual, predicted):.2f}")

AP@3: 0.50


Q9. The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [57]:
top3 = freq.index[:3].tolist()

scores = []

for actual in train["answer"]:
    scores.append(
        ap_at_3(actual, top3)
    )

overall_map3 = sum(scores) / len(scores)

print("MAP@3 =", overall_map3)

MAP@3 = 0.4212500000000017


Q10. The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  


In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

train = pd.read_csv("../dataset/smart-mcq-solver-challenge/train.csv")

# Fit TF-IDF on all prompt+option text
combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_text)

options = ["A", "B", "C", "D", "E"]

scores = []

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([str(row["prompt"])])

    similarities = []

    for opt in options:

        option_vec = vectorizer.transform([str(row[opt])])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        similarities.append((opt, sim))

    # Sort by similarity descending
    similarities.sort(key=lambda x: x[1], reverse=True)

    top3 = [x[0] for x in similarities[:3]]

    actual = row["answer"]

    if actual in top3:
        ap3 = 1.0 / (top3.index(actual) + 1)
    else:
        ap3 = 0.0

    scores.append(ap3)

map3 = sum(scores) / len(scores)

print("MAP@3 =", map3)
print("Rounded =", round(map3, 4))

MAP@3 = 0.29616666666666625
Rounded = 0.2962
